# 03a. Model Training: Decision Tree
Trains a **Decision Tree** (single tree) on the output of `featureEngineering.ipynb` (`processed/{train,val,test}/model_data.npz`).

**Task:** every row is one GPS point x one candidate arc; target = `on_route` (binary). The model outputs P(on_route), usable later as an HMM emission score.

**Protocol**
- Group-aware CV (`GroupKFold` by subject) on train + val subjects.
- Final fit on the `train` split.
- `test` = the 10 student-test trips, development comparison only. Evaluator data is never used.

Macro F1 here is over the two classes (on-route / off-route): arc IDs are local to each trip, so per-arc macro F1 is not meaningful.

## Imports and configuration

In [ ]:
import json, os, pickle, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score, log_loss,
                             precision_score, recall_score)
from sklearn.model_selection import GroupKFold
from sklearn.tree import DecisionTreeClassifier

MODEL_NAME = "DecisionTree"
# Resolve the dataset independently of VS Code/Jupyter's working directory.
from pathlib import Path
cwd = Path.cwd().resolve()
search_roots = [cwd, cwd / "Hackathon", cwd.parent, cwd.parent / "Hackathon"]
PROJECT_ROOT = next(
    (folder for folder in search_roots
     if (folder / "processingdata" / "train" / "model_data.npz").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        f"Cannot find processingdata/train/model_data.npz from {cwd}. "
        "Open the notebook in the Hackathon project or its parent folder."
    )
PROCESSED = str(PROJECT_ROOT / "processingdata")
OUT = str(PROJECT_ROOT / "outputs" / "decision_tree_baseline")
print("Reading dataset from:", PROCESSED)
print("Saving baseline results to:", OUT)
N_FOLDS = 5
SEED = 42
os.makedirs(OUT, exist_ok=True)
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40)

## Load data

In [ ]:
def load_split(split):
    z = np.load(os.path.join(PROCESSED, split, "model_data.npz"), allow_pickle=True)
    meta = pd.DataFrame({
        "subject_id": z["groups"].astype(str), "point_idx": z["point_idx"],
        "seg_id": z["seg_id"], "arc_id": z["arc_id"],
        "y": z["y"].astype(int), "y_true_arc": z["y_true_arc"].astype(int)})
    return z["X"], meta, [str(f) for f in z["feature_names"]]

Xtr, mtr, FEATURES = load_split("train")
Xva, mva, _ = load_split("val")
Xte, mte, _ = load_split("test")
print("train", Xtr.shape, "| val", Xva.shape, "| test", Xte.shape)
print("features:", FEATURES)

# CV pool = train + val subjects
Xcv = np.vstack([Xtr, Xva])
mcv = pd.concat([mtr, mva], ignore_index=True)
ycv, groups = mcv["y"].to_numpy(), mcv["subject_id"].to_numpy()

# leakage check: no subject shared between development data and student-test
assert not (set(mtr.subject_id) | set(mva.subject_id)) & set(mte.subject_id)
print("subjects  CV pool:", len(set(groups)), "| test:", mte.subject_id.nunique())
print("positive rate  train %.3f | val %.3f | test %.3f" % (mtr.y.mean(), mva.y.mean(), mte.y.mean()))

## Road networks (for route continuity)
Loaded from `interim.pkl`, which your own `dataProcessing.ipynb` wrote. If it is missing, route continuity is reported as N/A.

In [ ]:
pkl = os.path.join(PROCESSED, "interim", "interim.pkl")
networks = None
if os.path.exists(pkl):
    with open(pkl, "rb") as f:
        networks = pickle.load(f)["networks"]
else:
    print("interim.pkl not found -> route_continuity = N/A")

## Metric functions
- **Candidate-level:** accuracy, precision, recall, macro F1, log loss (threshold 0.5).
- **Point-level:** for each GPS point take the highest-scoring candidate.
  - `top1_on_route`: chosen arc is on the true route (lenient).
  - `top1_true_arc`: chosen arc equals `is_true_arc` (strict).
  - `candidate_miss_rate`: points with no on-route arc among their candidates.
  - `route_continuity`: among consecutive points of the same subject and segment, share of predicted arc pairs that are the same arc or end-to-start connected. It is conservative, because at 1 Hz a fast vehicle can skip a short arc.

In [ ]:
def candidate_metrics(y, p, thr=0.5):
    pred = (p >= thr).astype(int)
    return dict(
        accuracy=accuracy_score(y, pred),
        precision=precision_score(y, pred, zero_division=0),
        recall=recall_score(y, pred, zero_division=0),
        macro_f1=f1_score(y, pred, average="macro", zero_division=0),
        log_loss=log_loss(y, np.clip(p, 1e-7, 1 - 1e-7), labels=[0, 1]))


def point_metrics(meta, p):
    d = meta.assign(score=p).sort_values(["subject_id", "point_idx"]).reset_index(drop=True)
    g = d.groupby(["subject_id", "point_idx"], sort=False)
    top = d.loc[g["score"].idxmax()].reset_index(drop=True)
    has_true = g["y"].max().to_numpy()
    out = dict(n_points=len(top),
               top1_on_route=float(top["y"].mean()),
               top1_true_arc=float(top["y_true_arc"].mean()),
               candidate_miss_rate=float(1 - has_true.mean()),
               top1_on_route_given_hit=float(top.loc[has_true == 1, "y"].mean()))
    cont = np.nan
    if networks is not None:
        ok = tot = 0
        for sid, s in top.groupby("subject_id"):
            net = networks[sid]
            a, seg = s["arc_id"].to_numpy(), s["seg_id"].to_numpy()
            same = seg[1:] == seg[:-1]
            prev, cur = a[:-1][same], a[1:][same]
            conn = (net["canon_id"][prev] == net["canon_id"][cur]) | (net["arcs"][prev, 1] == net["arcs"][cur, 0])
            ok += int(conn.sum()); tot += int(len(conn))
        cont = ok / tot if tot else np.nan
    out["route_continuity"] = cont
    return out

## Model
A single CART tree. Its probabilities are **leaf class fractions**, so they are coarse (few distinct values) and log loss can be poor. Hyperparameters are reasonable starting values and are **not tuned**.

In [ ]:
def make_model():
    return DecisionTreeClassifier(max_depth=12, min_samples_leaf=50, random_state=SEED)

CONFIG_KEYS = ('max_depth', 'min_samples_leaf', 'random_state')

def proba_timed(model, X):
    t = time.perf_counter()
    p = model.predict_proba(X)[:, 1]
    return p, time.perf_counter() - t

## Group-aware cross-validation (train + val subjects)

In [ ]:
rows = []
for k, (tr, va) in enumerate(GroupKFold(n_splits=N_FOLDS).split(Xcv, ycv, groups)):
    m = make_model()
    m.fit(Xcv[tr], ycv[tr])
    p, _ = proba_timed(m, Xcv[va])
    r = candidate_metrics(ycv[va], p)
    r.update(point_metrics(mcv.iloc[va], p))
    r["fold"] = k
    rows.append(r)
    print(f"fold {k}: macroF1={r['macro_f1']:.4f}  top1={r['top1_on_route']:.4f}")
cv = pd.DataFrame(rows)
cv.to_csv(os.path.join(OUT, f"{MODEL_NAME}_cv_folds.csv"), index=False)
cv[["macro_f1", "top1_on_route", "log_loss"]].agg(["mean", "std"]).round(4)

## Final fit on `train`, evaluate on student-test

In [ ]:
model = make_model()
t = time.perf_counter()
model.fit(Xtr, mtr["y"].to_numpy())
train_time = time.perf_counter() - t
p_te, inf_time = proba_timed(model, Xte)
te = candidate_metrics(mte["y"].to_numpy(), p_te)
te.update(point_metrics(mte, p_te))

cm = confusion_matrix(mte["y"].to_numpy(), (p_te >= 0.5).astype(int))
np.savetxt(os.path.join(OUT, f"{MODEL_NAME}_confusion_test.csv"), cm, fmt="%d", delimiter=",")
np.save(os.path.join(OUT, f"{MODEL_NAME}_test_scores.npy"), p_te.astype(np.float32))
with open(os.path.join(OUT, f"{MODEL_NAME}.pkl"), "wb") as f:
    pickle.dump(model, f)

summary = dict(
    Model=MODEL_NAME,
    Key_configuration=json.dumps({k: v for k, v in model.get_params().items() if k in CONFIG_KEYS}),
    CV_macro_f1=cv.macro_f1.mean(), CV_macro_f1_std=cv.macro_f1.std(),
    CV_top1=cv.top1_on_route.mean(), CV_log_loss=cv.log_loss.mean(),
    Test_macro_f1=te["macro_f1"], Test_accuracy=te["accuracy"],
    Test_precision=te["precision"], Test_recall=te["recall"],
    Test_top1_on_route=te["top1_on_route"], Test_top1_true_arc=te["top1_true_arc"],
    Test_candidate_miss_rate=te["candidate_miss_rate"],
    Test_route_continuity=te["route_continuity"], Test_log_loss=te["log_loss"],
    Train_time_s=train_time, Inference_time_s=inf_time,
    Inference_us_per_row=1e6 * inf_time / len(Xte), seed=SEED)
pd.DataFrame([summary]).to_csv(os.path.join(OUT, f"{MODEL_NAME}_summary.csv"), index=False)
pd.Series(summary).drop("Key_configuration").to_frame("value").round(4)

## Plots (from the measured results only)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
axes[0].bar(cv["fold"].astype(str), cv["macro_f1"]); axes[0].set_title("CV Macro F1 per fold")
axes[0].set_xlabel("fold"); axes[0].set_ylim(0, 1)
axes[1].imshow(cm, cmap="Blues")
for (i, j), v in np.ndenumerate(cm):
    axes[1].text(j, i, f"{v:,}", ha="center", va="center")
axes[1].set_title("Student-test confusion matrix"); axes[1].set_xlabel("predicted"); axes[1].set_ylabel("actual")
axes[1].set_xticks([0, 1]); axes[1].set_yticks([0, 1])
plt.tight_layout(); plt.savefig(os.path.join(OUT, f"{MODEL_NAME}_plots.png"), dpi=150); plt.show()
imp = pd.Series(model.feature_importances_, index=FEATURES).sort_values()
plt.figure(figsize=(5.5, 5)); imp.plot.barh(); plt.title("Decision Tree feature importance (impurity)")
plt.tight_layout(); plt.savefig(os.path.join(OUT, f"{MODEL_NAME}_feature_importance.png"), dpi=150); plt.show()

## Notes for the report
- A decision tree has no epoch-wise history, so there is no loss curve for it.
- Probabilities are leaf fractions (coarse); mention this when you discuss its log loss.
- The scaler was fit on `train` only while the CV pool includes `val`; this has no effect on tree models.
- Report only the numbers produced by running this notebook.

---
## Optional tuning addendum — original notebook cells unchanged
This independent section tunes the Decision Tree using **group-aware cross-validation** on the existing `train` and `val` candidate data, and then evaluates once on the student-test split. It uses the updated `Hackathon/processingdata` structure and reads trusted local NumPy object arrays with `allow_pickle=True`.

**Run only the cells in this addendum** if the original notebook's older `processed` path causes errors. The original 18 cells have not been edited or removed. All new outputs are saved to `outputs/decision_tree_tuned` so the original baseline files are not overwritten.


In [ ]:
# Independent tuning setup; existing notebook cells are unchanged.
import json
import time
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GroupKFold, RandomizedSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, log_loss, confusion_matrix

cwd = Path.cwd().resolve()
search_roots = [cwd, cwd / "Hackathon", cwd.parent, cwd.parent / "Hackathon"]
ROOT_TUNED = next((p for p in search_roots if (p / "processingdata" / "train" / "model_data.npz").is_file()), None)
if ROOT_TUNED is None:
    raise FileNotFoundError(f"Cannot locate processingdata/train/model_data.npz from {cwd}")
DATA_TUNED = ROOT_TUNED / "processingdata"
OUT_TUNED = ROOT_TUNED / "outputs" / "decision_tree_tuned"
OUT_TUNED.mkdir(parents=True, exist_ok=True)

# Only load NPZ files you generated yourself: object arrays require pickle support.
def load_tuned(split):
    path = DATA_TUNED / split / "model_data.npz"
    with np.load(path, allow_pickle=True) as z:
        required = ["X", "y", "groups", "point_idx", "arc_id", "y_true_arc", "feature_names"]
        missing = [key for key in required if key not in z.files]
        if missing:
            raise KeyError(f"{path} missing {missing}")
        x = np.asarray(z["X"], dtype=np.float32)
        y = np.asarray(z["y"], dtype=np.int8)
        groups = np.asarray(z["groups"], dtype=str)
        point_idx = np.asarray(z["point_idx"])
        true_arc = np.asarray(z["y_true_arc"], dtype=np.int8)
        feature_names = [str(f) for f in z["feature_names"]]
    assert x.ndim == 2 and len(x) == len(y) == len(groups) == len(point_idx) == len(true_arc)
    assert x.shape[1] == len(feature_names)
    assert set(np.unique(y)).issubset({0, 1})
    return dict(X=x, y=y, groups=groups, point_idx=point_idx, true_arc=true_arc, features=feature_names)

TR_TUNED = load_tuned("train")
VA_TUNED = load_tuned("val")
TE_TUNED = load_tuned("test")
assert TR_TUNED["features"] == VA_TUNED["features"] == TE_TUNED["features"]
assert not (set(TR_TUNED["groups"]) | set(VA_TUNED["groups"])) & set(TE_TUNED["groups"])
X_DEV = np.concatenate([TR_TUNED["X"], VA_TUNED["X"]])
Y_DEV = np.concatenate([TR_TUNED["y"], VA_TUNED["y"]])
GROUP_DEV = np.concatenate([TR_TUNED["groups"], VA_TUNED["groups"]])
SEED_TUNED = 42
N_TRIALS = 15  # increase to 30 for a broader search if time permits
CV_SPLITS = min(3, len(set(GROUP_DEV)))
if CV_SPLITS < 2:
    raise ValueError("Need at least 2 unique training subjects for group-aware CV")
print("Project:", ROOT_TUNED)
print("Training:", TR_TUNED["X"].shape, "validation:", VA_TUNED["X"].shape, "student test:", TE_TUNED["X"].shape)
print("Distinct development subjects:", len(set(GROUP_DEV)))
print("Output:", OUT_TUNED)


In [ ]:
# Tune macro-F1 with subject-grouped CV; do NOT touch student-test during selection.
param_dist = {
    "max_depth": [5, 7, 9, 12, 15, 20, None],
    "min_samples_leaf": [10, 20, 35, 50, 75, 100, 150],
    "min_samples_split": [2, 10, 25, 50, 100],
    "max_features": [None, "sqrt", 0.75],
    "criterion": ["gini", "entropy"],
    "ccp_alpha": [0.0, 0.00001, 0.0001, 0.001],
}
search = RandomizedSearchCV(
    estimator=DecisionTreeClassifier(random_state=SEED_TUNED),
    param_distributions=param_dist,
    n_iter=N_TRIALS,
    scoring="f1_macro",
    cv=GroupKFold(n_splits=CV_SPLITS),
    random_state=SEED_TUNED,
    n_jobs=-1,
    refit=False,
    verbose=1,
    error_score="raise",
)
start = time.perf_counter()
search.fit(X_DEV, Y_DEV, groups=GROUP_DEV)
search_seconds = time.perf_counter() - start
print("Best cross-validation macro F1:", round(search.best_score_, 4))
print("Best parameters:", search.best_params_)
print("Search seconds:", round(search_seconds, 1))
cv_results = pd.DataFrame(search.cv_results_).sort_values("rank_test_score")
cv_results.to_csv(OUT_TUNED / "tuning_cv_results.csv", index=False)
(OUT_TUNED / "best_parameters.json").write_text(json.dumps(search.best_params_, indent=2), encoding="utf-8")


In [ ]:
# Fit the tuned model on TRAIN only to preserve original baseline protocol;
# compare against original max_depth=12, min_samples_leaf=50 on the same student-test data.
def assess_tuned(name, clf):
    tic = time.perf_counter()
    clf.fit(TR_TUNED["X"], TR_TUNED["y"])
    fit_seconds = time.perf_counter() - tic
    p = clf.predict_proba(TE_TUNED["X"])[:, list(clf.classes_).index(1)]
    pred = (p >= 0.5).astype(np.int8)
    y = TE_TUNED["y"]
    result = {
        "model": name,
        "accuracy": float(accuracy_score(y, pred)),
        "precision": float(precision_score(y, pred, zero_division=0)),
        "recall": float(recall_score(y, pred, zero_division=0)),
        "macro_f1": float(f1_score(y, pred, average="macro", zero_division=0)),
        "log_loss": float(log_loss(y, np.clip(p, 1e-7, 1-1e-7), labels=[0, 1])),
        "training_seconds": round(fit_seconds, 2),
    }
    points = pd.DataFrame({
        "subject": TE_TUNED["groups"], "point": TE_TUNED["point_idx"],
        "score": p, "on_route": y, "true_arc": TE_TUNED["true_arc"],
    })
    winners = points.loc[points.groupby(["subject", "point"], sort=False)["score"].idxmax()]
    result["top1_on_route"] = float(winners["on_route"].mean())
    result["top1_true_arc"] = float(winners["true_arc"].mean())
    return result, clf, confusion_matrix(y, pred), p

baseline_result, baseline_model, _, _ = assess_tuned(
    "DecisionTree_baseline", DecisionTreeClassifier(max_depth=12, min_samples_leaf=50, random_state=SEED_TUNED))
tuned_result, tuned_model, tuned_cm, tuned_scores = assess_tuned(
    "DecisionTree_tuned", DecisionTreeClassifier(**search.best_params_, random_state=SEED_TUNED))
comparison = pd.DataFrame([baseline_result, tuned_result]).set_index("model")
display(comparison.round(4))
comparison.to_csv(OUT_TUNED / "baseline_vs_tuned_student_test.csv")
joblib.dump(tuned_model, OUT_TUNED / "decision_tree_tuned.joblib")
np.save(OUT_TUNED / "tuned_student_test_scores.npy", tuned_scores.astype(np.float32))
np.savetxt(OUT_TUNED / "tuned_confusion_matrix.csv", tuned_cm, fmt="%d", delimiter=",")
feature_imp = pd.Series(tuned_model.feature_importances_, index=TR_TUNED["features"]).sort_values(ascending=False)
feature_imp.to_csv(OUT_TUNED / "feature_importance.csv", header=["importance"])
fig, ax = plt.subplots(figsize=(7, 5))
feature_imp.sort_values().plot.barh(ax=ax)
ax.set_title("Tuned Decision Tree feature importance")
fig.tight_layout()
fig.savefig(OUT_TUNED / "feature_importance.png", dpi=160)
plt.show()
print("Saved tuned model and evaluation files to:", OUT_TUNED)
print("Note: the student test is for final comparison only, not tuning or selection.")
